# Phase 2 — Instagram 해시태그 수집 (A안: IG Hashtag Search API)

공개 SNS VOC 보강을 위한 **A안** — Instagram Graph API의 **Hashtag Search**로 해시태그별 공개 게시물(캡션)을 수집합니다.

## ⚠️ 전제 조건 (없으면 실행 불가)
Instagram Graph API는 **익명/공개 접근이 불가**합니다. 아래가 모두 필요합니다.
1. **Instagram 비즈니스/크리에이터 계정** + 그것과 연결된 **Facebook 페이지**
2. 그 페이지를 관리하는 **장기(long-lived) 액세스 토큰** — 권한 `instagram_basic`, `pages_show_list`, `pages_read_engagement`
3. **IG 비즈니스 계정 ID**(`ig_user_id`)

`.env`에 추가:
```
FACEBOOK_ACCESS_TOKEN=<long-lived 토큰>
IG_USER_ID=<IG 비즈니스 계정 ID>      # 없으면 아래 셀에서 FB 페이지로부터 자동 조회
FB_PAGE_ID=<연결된 FB 페이지 ID>       # IG_USER_ID 자동 조회용(선택)
```

## ⚠️ API 한계 (수집 전 반드시 인지)
- **30개 해시태그 / 7일** 제한 (IG 사용자당).
- `recent_media` ≈ **최근 24시간** 게시물만, `top_media` = 인기 게시물(롤링).
- **캡션·좋아요/댓글 수만** 반환 → **댓글 본문·작성자 정보는 제공 안 됨**(프라이버시).
- 해시태그는 `#`·공백 없이 단일 토큰만 검색 가능.
- 라오스 EV 관련 해시태그는 **절대량이 적어 결과가 희소**할 수 있음 → 그 경우 B안(앱리뷰·유튜브) 유지가 합리적.


In [ ]:
# !pip install requests psycopg2-binary pandas python-dotenv

In [ ]:
import sys, os, time
sys.path.append('../')
import requests
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from src.db import insert_df   # sns_posts 적재용

load_dotenv(dotenv_path='../.env')

API_VERSION = 'v21.0'
BASE = f'https://graph.facebook.com/{API_VERSION}'
TOKEN   = os.getenv('FACEBOOK_ACCESS_TOKEN')
IG_USER = os.getenv('IG_USER_ID')
FB_PAGE = os.getenv('FB_PAGE_ID')

assert TOKEN, '❌ .env에 FACEBOOK_ACCESS_TOKEN 필요'
print('토큰 로드 OK, 길이:', len(TOKEN))

## 0) IG 비즈니스 계정 ID 확인
`IG_USER_ID`가 없으면, 연결된 FB 페이지에서 자동 조회합니다.


In [ ]:
def resolve_ig_user_id():
    if IG_USER:
        return IG_USER
    # FB 페이지에 연결된 IG 비즈니스 계정 조회
    if FB_PAGE:
        r = requests.get(f'{BASE}/{FB_PAGE}', params={
            'fields': 'instagram_business_account', 'access_token': TOKEN}).json()
        return r.get('instagram_business_account', {}).get('id')
    # 토큰으로 관리 페이지 목록 → 첫 IG 계정
    r = requests.get(f'{BASE}/me/accounts', params={'access_token': TOKEN}).json()
    for pg in r.get('data', []):
        rr = requests.get(f'{BASE}/{pg["id"]}', params={
            'fields': 'instagram_business_account', 'access_token': TOKEN}).json()
        iga = rr.get('instagram_business_account', {}).get('id')
        if iga:
            print('연결된 IG 계정 발견:', pg.get('name'), iga); return iga
    return None

ig_user_id = resolve_ig_user_id()
print('IG_USER_ID =', ig_user_id)
assert ig_user_id, '❌ IG 비즈니스 계정 ID를 확인하지 못했습니다 (.env IG_USER_ID 또는 FB_PAGE_ID 설정 필요)'

## 1) 수집 대상 해시태그
라오스·베트남·태국 EV 충전 관련. `#`·공백 없이. (30개/7일 제한 → 핵심만)


In [ ]:
# (hashtag, country) — country는 CHAR(3): LAO/VNM/THA/MUL
HASHTAGS = [
    ('locaev',        'LAO'),
    ('evlaos',        'LAO'),
    ('ລົດໄຟຟ້າ',       'LAO'),   # 라오어 '전기차'
    ('vientianeev',   'LAO'),
    ('vinfast',       'VNM'),
    ('vgreen',        'VNM'),
    ('xedien',        'VNM'),   # 베트남어 '전기차'
    ('evcharging',    'MUL'),
    ('evchargingstation','MUL'),
    ('รถยนต์ไฟฟ้า',    'THA'),   # 태국어 '전기차'
]
print(len(HASHTAGS), '개 해시태그')

In [ ]:
def search_hashtag_id(tag):
    r = requests.get(f'{BASE}/ig_hashtag_search', params={
        'user_id': ig_user_id, 'q': tag, 'access_token': TOKEN}).json()
    data = r.get('data', [])
    if not data:
        print(f'  [{tag}] 해시태그 없음/조회 실패: {r.get("error",{}).get("message","")}')
        return None
    return data[0]['id']

def get_media(hashtag_id, edge='recent_media', limit=50):
    """edge: recent_media(최근 24h) | top_media(인기)"""
    fields = 'id,caption,media_type,permalink,timestamp,like_count,comments_count'
    out, url = [], f'{BASE}/{hashtag_id}/{edge}'
    params = {'user_id': ig_user_id, 'fields': fields, 'limit': limit, 'access_token': TOKEN}
    while url and len(out) < limit:
        r = requests.get(url, params=params).json()
        out.extend(r.get('data', []))
        url = r.get('paging', {}).get('next'); params = {}  # next는 풀 URL
    return out

## 2) 수집 실행


In [ ]:
rows = []
for tag, country in HASHTAGS:
    hid = search_hashtag_id(tag)
    if not hid:
        continue
    for edge in ('recent_media', 'top_media'):
        for m in get_media(hid, edge):
            cap = (m.get('caption') or '').strip()
            if not cap:
                continue  # 캡션 없는 게시물 제외
            ts = m.get('timestamp', '')[:10] or None
            rows.append({
                'platform': 'instagram',
                'page_name': f'#{tag}',
                'country': country,
                'content': cap,
                'post_date': ts,
                'collection_method': 'graph_api',
            })
    time.sleep(1)  # rate-limit 여유

print('원시 수집:', len(rows), '건')

## 3) 정제 → DataFrame (sns_posts 스키마)


In [ ]:
df = pd.DataFrame(rows)
if not df.empty:
    df = df.drop_duplicates(subset=['content']).reset_index(drop=True)  # 동일 캡션 중복 제거
    # content 최소 길이 필터 (해시태그만 있는 글 제외)
    df = df[df['content'].str.len() >= 10].reset_index(drop=True)
print('정제 후:', len(df), '건')
df.head(10)

## 4) DB 적재 (sns_posts)
검수 후 주석 해제하여 실행. 언어감지·감성분석·키워드분류는 Phase 3 전처리 노트북에서 일괄 적용.


In [ ]:
# insert_df(df, 'sns_posts')   # ← 데이터 확인 후 주석 해제
print('적재 준비 완료 — 위 주석 해제 시 sns_posts 삽입')

## 5) 한계 & 다음 단계
- **결과가 희소하면**: 라오스 IG 해시태그 절대량 부족 → 정량 분석엔 부적합. 앱리뷰(28,890)·유튜브 중심 유지(B안).
- **댓글 본문 불가**: Hashtag API는 캡션·집계만. 사용자 반응 정성 분석은 유튜브 댓글로 대체.
- **다음**: 수집되면 Phase 3 전처리(langdetect → XLM-RoBERTa 감성 → 키워드)로 `sns_posts`의 `lang_detected/sentiment_*/keyword_category` 채움.
